# Ball Detection — Hough Circles

## Imports

In [ ]:
import os
import cv2
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Load Dataset

In [ ]:
DATASET_DIR = "development_set/"

In [ ]:
image_paths = sorted([
    os.path.join(DATASET_DIR, f)
    for f in os.listdir(DATASET_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print(f"Found {len(image_paths)} images")

In [ ]:
def show_images(images, titles=None, max_cols=4, figsize_per_image=(4, 3)):
    n = len(images)
    if n == 0:
        print("No images to display.")
        return

    cols = min(n, max_cols)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per_image[0] * cols,
                                                   figsize_per_image[1] * rows))
    axes = np.array(axes).flatten()

    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, str):
                img = cv2.imread(img)

            if img is not None:
                if img.ndim == 2:                          # ← grayscale / mask
                    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
                else:                                      # ← colour image
                    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

            ax.set_title(titles[i] if titles and i < len(titles) else f"Image {i+1}",
                         fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Ball Detection

In [ ]:
def preprocess_lighting(hsv_image):
    # 1. Split the HSV image into its three separate channels
    h, s, v = cv2.split(hsv_image)

    # 2. Create the CLAHE filter
    # clipLimit prevents noise from being amplified too much
    # tileGridSize is the size of the localized "checkerboard" squares
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    # 3. Apply the filter ONLY to the V (brightness) channel
    v_eq = clahe.apply(v)

    # 4. Merge the channels back together
    hsv_eq = cv2.merge((h, s, v_eq))

    return hsv_eq

def get_table_mask(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv = preprocess_lighting(hsv)
    h, w = image.shape[:2]

    # Sample using median to avoid balls in the center
    # 1. Define 5 safe sampling points (Center + 4 inner quadrants)
    points = [
        (h//2, w//2),               # Center
        (int(h*0.4), int(w*0.4)),   # Top-Left inner
        (int(h*0.4), int(w*0.6)),   # Top-Right inner
        (int(h*0.6), int(w*0.4)),   # Bottom-Left inner
        (int(h*0.6), int(w*0.6))    # Bottom-Right inner
    ]

    # 2. Collect a 40x40 patch from ALL 5 locations
    samples = []
    for py, px in points:
        patch = hsv[py-20:py+20, px-20:px+20]
        samples.append(patch)

    # 3. Stack all 8,000 pixels together and find the true median
    all_samples = np.vstack(samples)
    median_hsv = np.median(all_samples, axis=(0, 1))

    # Build tolerance and threshold
    tol = np.array([15, 120, 120])
    lower = np.clip(median_hsv - tol, 0, 255).astype(np.uint8)
    upper = np.clip(median_hsv + tol, 0, 255).astype(np.uint8)

    return cv2.inRange(hsv, lower, upper)

def isolate_largest_blob(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return np.zeros_like(mask)

    largest_contour = max(contours, key=cv2.contourArea)

    clean_mask = np.zeros_like(mask)

    cv2.drawContours(clean_mask, [largest_contour], -1, 255, thickness=cv2.FILLED)

    return clean_mask

In [ ]:
images = [cv2.imread(p) for p in image_paths]
masks  = [get_table_mask(img) for img in images]

show_images(masks, titles=[os.path.basename(p) for p in image_paths],max_cols=3)

In [ ]:
def detect_blue_balls(img, existing_circles=None):
    if existing_circles is None:
        existing_circles = []

    table_mask        = get_table_mask(img)
    playing_area_mask = isolate_largest_blob(table_mask)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hsv = preprocess_lighting(hsv)

    h, w = img.shape[:2]
    pts  = [
        (h//2, w//2),
        (int(h*0.4), int(w*0.4)), (int(h*0.4), int(w*0.6)),
        (int(h*0.6), int(w*0.4)), (int(h*0.6), int(w*0.6)),
    ]
    patches   = [hsv[py-20:py+20, px-20:px+20] for py, px in pts]
    table_hsv = np.median(np.vstack(patches), axis=(0, 1))

    if existing_circles:
        median_r = int(np.median([r for _, _, r in existing_circles]))
    else:
        median_r = 18

    r_min = max(8,  int(median_r * 0.70))
    r_max = min(45, int(median_r * 1.35))

    play_img  = cv2.bitwise_and(img, img, mask=playing_area_mask)
    bilateral = cv2.bilateralFilter(play_img, d=9, sigmaColor=75, sigmaSpace=75)
    gray      = cv2.cvtColor(bilateral, cv2.COLOR_BGR2GRAY)

    raw = cv2.HoughCircles(
        gray,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=int(median_r * 1.5),
        param1=40,
        param2=18,
        minRadius=r_min,
        maxRadius=r_max,
    )

    if raw is None:
        return [], []

    candidates = np.round(raw[0]).astype(int)
    blue_circles = []
    blue_bboxes = []

    for (cx, cy, r) in candidates:
        if playing_area_mask[cy, cx] == 0:
            continue

        overlap = any(
            np.hypot(cx - ex, cy - ey) < (r + er) * 0.6
            for (ex, ey, er) in existing_circles
        )
        if overlap:
            continue

        inner_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        cv2.circle(inner_mask, (cx, cy), max(r // 2, 4), 255, -1)
        ball_pixels = hsv[inner_mask == 255]
        if len(ball_pixels) < 5:
            continue

        ball_h, ball_s, ball_v = np.median(ball_pixels, axis=0)

        if ball_v < 50: continue
        if 35 <= ball_h <= 85: continue
        if 130 <= ball_h <= 170: continue
        if not (85 <= ball_h <= 130): continue

        hue_diff = min(abs(ball_h - table_hsv[0]), 180 - abs(ball_h - table_hsv[0]))
        val_diff = float(table_hsv[2]) - float(ball_v)
        sat_diff = float(ball_s) - float(table_hsv[1])

        if hue_diff < 25 and val_diff < 10 and sat_diff < 20:
            continue

        final_r = int(r * 1.1)
        blue_circles.append((cx, cy, final_r))
        # Calculate bounding box: (x, y, w, h)
        blue_bboxes.append((cx - final_r, cy - final_r, 2 * final_r, 2 * final_r))

    return blue_circles, blue_bboxes

In [ ]:
processed_images = []
image_titles = []

print(f"Processing {len(image_paths)} images for BLUE balls specifically...")

for path in image_paths:
    img = cv2.imread(path)
    if img is not None:
        # Call the specific blue ball function.
        # Passing None to existing_circles so it looks for all blue balls.
        blue_circles, blue_bboxes = detect_blue_balls(img, existing_circles=None)

        result_img = img.copy()

        # 1. Draw Bounding Boxes (Red in BGR is (0, 0, 255))
        for (x, y, w, h) in blue_bboxes:
            cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 0, 255), 2)

        # 2. Draw Circles (Blue in BGR is (255, 0, 0)) and Center Dots
        for (cx, cy, r) in blue_circles:
            cv2.circle(result_img, (cx, cy), r, (255, 0, 0), 3)
            cv2.circle(result_img, (cx, cy), 1, (0, 255, 0), 2)

        processed_images.append(result_img)
        filename = os.path.basename(path)
        image_titles.append(f"{filename} ({len(blue_circles)} blue)")

# Display the results specifically for blue balls
show_images(processed_images, titles=image_titles, max_cols=3, figsize_per_image=(8, 6))

In [ ]:
def detect_balls(img):
    """
    Receives a BGR image array.
    Returns (circles, bboxes)
    circles: [(cx, cy, r), ...]
    bboxes: [(x, y, w, h), ...]
    """
    table_mask = get_table_mask(img)
    playing_area_mask = isolate_largest_blob(table_mask)

    not_table_mask = cv2.bitwise_not(table_mask)
    balls_mask = cv2.bitwise_and(not_table_mask, playing_area_mask)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    balls_mask = cv2.morphologyEx(balls_mask, cv2.MORPH_OPEN, kernel, iterations=1)

    dist_transform = cv2.distanceTransform(balls_mask, cv2.DIST_L2, 5)

    if dist_transform.max() == 0:
        return [], []

    blobs, _ = cv2.findContours(balls_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    circles = []
    bboxes = []

    for blob in blobs:
        blob_mask = np.zeros_like(balls_mask)
        cv2.drawContours(blob_mask, [blob], -1, 255, thickness=cv2.FILLED)

        blob_dist = np.zeros_like(dist_transform)
        blob_dist[blob_mask == 255] = dist_transform[blob_mask == 255]

        max_val = blob_dist.max()
        if max_val == 0:
            continue

        _, peaks = cv2.threshold(blob_dist, 0.6 * max_val, 255, 0)
        peaks = np.uint8(peaks)

        peak_contours, _ = cv2.findContours(peaks, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in peak_contours:
            M = cv2.moments(cnt)
            if M["m00"] > 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])

                radius = int(dist_transform[cy, cx])

                if 8 < radius < 45:
                    final_r = int(radius * 1.1)
                    circles.append((cx, cy, final_r))
                    # Calculate bounding box: (x, y, w, h)
                    bboxes.append((cx - final_r, cy - final_r, 2 * final_r, 2 * final_r))

    # Get blue balls
    blue_circles, blue_bboxes = detect_blue_balls(img, existing_circles=circles)

    circles.extend(blue_circles)
    bboxes.extend(blue_bboxes)

    return circles, bboxes

In [ ]:
processed_images_circles = []
image_titles_circles = []

print("Processing images for Circles only...")

for path in image_paths:
    img = cv2.imread(path)
    if img is not None:
        # Unpack both, but we only use circles here
        circles, _ = detect_balls(img)

        result_img = img.copy()

        for (cx, cy, r) in circles:
            # Draw the circle outline (Red)
            cv2.circle(result_img, (cx, cy), r, (0, 0, 255), 3)
            # Draw a bright center dot (Green) for precision check
            cv2.circle(result_img, (cx, cy), 1, (0, 255, 0), 2)

        processed_images_circles.append(result_img)
        filename = os.path.basename(path)
        image_titles_circles.append(f"{filename} - {len(circles)} Circles")

show_images(processed_images_circles, titles=image_titles_circles, max_cols=3, figsize_per_image=(8, 6))

In [ ]:
processed_images_boxes = []
image_titles_boxes = []

print("Processing images for Bounding Boxes only...")

for path in image_paths:
    img = cv2.imread(path)
    if img is not None:
        # Unpack both, but we only use bboxes here
        _, bboxes = detect_balls(img)

        result_img = img.copy()

        for (x, y, w, h) in bboxes:
            # Draw the Bounding Box (Green)
            cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 255, 0), 3)

        processed_images_boxes.append(result_img)
        filename = os.path.basename(path)
        image_titles_boxes.append(f"{filename} - {len(bboxes)} Boxes")

show_images(processed_images_boxes, titles=image_titles_boxes, max_cols=3, figsize_per_image=(8, 6))

In [ ]:
def evaluate_detector(csv_path, dataset_dir):
    # 1. Read the CSV file using pandas
    df = pd.read_csv(csv_path, header=None, names=['filename', 'true_count'])

    # Clean up the data
    df = df[df['true_count'].apply(lambda x: str(x).isdigit())]
    df['true_count'] = df['true_count'].astype(int)

    errors = []
    exact_matches = 0
    results = []

    print(f"Evaluating {len(df)} images, please wait...")

    # 2. Loop through the rows
    for index, row in df.iterrows():
        filename = str(row['filename']).strip()
        true_count = row['true_count']

        image_path = os.path.join(dataset_dir, filename)
        if not os.path.exists(image_path):
            print(f"Warning: File not found - {filename}")
            continue

        img = cv2.imread(image_path)
        if img is None:
            print(f"Warning: Could not read image - {filename}")
            continue

        # 3. Predict the number of balls
        circles,bboxes = detect_balls(img)
        pred_count = len(circles)

        # 4. Calculate absolute error for this image
        error = abs(true_count - pred_count)
        errors.append(error)

        if error == 0:
            exact_matches += 1

        results.append({
            'Filename': filename,
            'True Count': true_count,
            'Predicted': pred_count,
            'Error (Missed)': error
        })

    if len(results) == 0:
        print("No valid images processed. Check your CSV path and dataset directory.")
        return

    # 5. Calculate Dataset-Wide Metrics
    results_df = pd.DataFrame(results)

    total_images = len(results_df)
    total_true_balls = results_df['True Count'].sum()
    total_pred_balls = results_df['Predicted'].sum()
    total_absolute_error = results_df['Error (Missed)'].sum()

    # Image-Level Metrics
    mae_per_image = total_absolute_error / total_images
    exact_match_accuracy = (exact_matches / total_images) * 100

    # Entire Dataset Overall Error Rate
    dataset_error_rate = (total_absolute_error / total_true_balls) * 100 if total_true_balls > 0 else 0

    # 6. Print the beautiful summary
    print("\n" + "="*50)
    print("           FULL DATASET EVALUATION")
    print("="*50)
    print(f"Images Evaluated:          {total_images}")
    print(f"Total True Balls:          {total_true_balls}")
    print(f"Total Predicted Balls:     {total_pred_balls}")
    print(f"Total Balls Missed:        {total_absolute_error}")
    print("-" * 50)
    print(f"Image-Level MAE:           {mae_per_image:.3f} (avg balls missed/image)")
    print(f"Exact Match Accuracy:      {exact_match_accuracy:.2f}% (images with 0 errors)")
    print(f"Overall Dataset Error:     {dataset_error_rate:.2f}% (missed / true total)")
    print("="*50 + "\n")

    # 7. Sort the dataframe to show the worst errors
    results_df = results_df.sort_values(by='Error (Missed)', ascending=False).reset_index(drop=True)

    print("--- Top 5 Worst Errors ---")
    display(results_df.head(5))

    return results_df

In [ ]:
my_results = evaluate_detector("ball_count.csv", DATASET_DIR)